# AS-STUDIO-PRO V2.1
Run Step 1 then Step 2.

In [ ]:
# STEP 1: SETUP DEPENDENCIES
!apt-get update -qq && apt-get install -y -qq ffmpeg wget
!pip install -q omnivoice gradio soundfile "librosa>=0.10.0" ffmpeg-python moviepy opencv-python torch torchvision tqdm huggingface_hub

import os
import shutil
from huggingface_hub import hf_hub_download

HF_TOKEN = "" 

if not os.path.exists('/content/Wav2Lip'):
    !git clone https://github.com/Rudrabha/Wav2Lip.git

%cd /content/Wav2Lip
!mkdir -p face_detection/detection checkpoints temp results

# Fix Librosa
audio_py_path = "/content/Wav2Lip/audio.py"
if os.path.exists(audio_py_path):
    with open(audio_py_path, "r") as f:
        content = f.read()
    old_code = "return librosa.filters.mel(hp.sample_rate, hp.n_fft, n_mels=hp.num_mels,"
    new_code = "return librosa.filters.mel(sr=hp.sample_rate, n_fft=hp.n_fft, n_mels=hp.num_mels,"
    if old_code in content:
        with open(audio_py_path, "w") as f:
            f.write(content.replace(old_code, new_code))

# Fix PyTorch load
inference_py_path = "/content/Wav2Lip/inference.py"
if os.path.exists(inference_py_path):
    with open(inference_py_path, "r") as f:
        inf_content = f.read()
    if "weights_only" not in inf_content:
        inf_content = inf_content.replace(
            "checkpoint = torch.load(checkpoint_path)",
            "checkpoint = torch.load(checkpoint_path, map_location=lambda storage, loc: storage, weights_only=False)"
        )
        with open(inference_py_path, "w") as f:
            f.write(inf_content)

if not os.path.exists("face_detection/detection/s3fd-619a5ba8.pth"):
    !wget -q "https://www.adrianbulat.com/downloads/python-s3fd/s3fd-619a5ba8.pth" -O "face_detection/detection/s3fd-619a5ba8.pth"

ckpt_path = "checkpoints/wav2lip_gan.pth"
if os.path.exists(ckpt_path) and os.path.getsize(ckpt_path) < 400 * 1024 * 1024:
    os.remove(ckpt_path)

if not os.path.exists(ckpt_path):
    try:
        downloaded_file = hf_hub_download(repo_id="camenduru/Wav2Lip", filename="checkpoints/wav2lip_gan.pth", token=HF_TOKEN if HF_TOKEN else None)
        shutil.copy(downloaded_file, ckpt_path)
    except Exception as e:
        print(f"[!] Mirror download failed: {e}")

size_mb = os.path.getsize(ckpt_path) / (1024 * 1024) if os.path.exists(ckpt_path) else 0
print(f"\n✅ SETUP COMPLETE! Checkpoint Size: {size_mb:.2f} MB")


In [ ]:
# STEP 2: LAUNCH GRADIO WEB UI
import gradio as gr
import subprocess
import os

def process_lip_sync(video_path, audio_path, aspect_ratio):
    if not video_path or not audio_path:
        return None
    
    os.chdir("/content/Wav2Lip")
    os.makedirs("/content/Wav2Lip/temp", exist_ok=True)
    os.makedirs("/content/Wav2Lip/results", exist_ok=True)
    
    raw_output = "/content/Wav2Lip/results/result_raw.mp4"
    final_output = "/content/Wav2Lip/results/result_final.mp4"
    
    if os.path.exists(raw_output):
        os.remove(raw_output)
    if os.path.exists(final_output):
        os.remove(final_output)

    cmd = [
        "python", "inference.py",
        "--checkpoint_path", "checkpoints/wav2lip_gan.pth",
        "--face", video_path,
        "--audio", audio_path,
        "--outfile", raw_output,
        "--pads", "0", "10", "0", "0",
        "--resize_factor", "1"
    ]
    
    print("[+] Processing Lip-Sync Video...")
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd="/content/Wav2Lip")
    for line in process.stdout:
        print(line, end="")
    process.wait()
    
    if not os.path.exists(raw_output):
        print("❌ Error: Wav2Lip failed to create video.")
        return None

    crop_filter = None
    if aspect_ratio == "16:9":
        crop_filter = "crop=w='min(iw,ih*16/9)':h='min(ih,iw*9/16)':x='(iw-ow)/2':y='(ih-oh)/2',scale=1920:1080"
    elif aspect_ratio == "9:16":
        crop_filter = "crop=w='min(iw,ih*9/16)':h='min(ih,iw*16/9)':x='(iw-ow)/2':y='(ih-oh)/2',scale=1080:1920"
    elif aspect_ratio == "1:1":
        crop_filter = "crop=w='min(iw,ih)':h='min(iw,ih)':x='(iw-ow)/2':y='(ih-oh)/2',scale=1080:1080"

    if crop_filter:
        print("[+] Formatting Aspect Ratio...")
        ffmpeg_cmd = ["ffmpeg", "-y", "-i", raw_output, "-vf", crop_filter, "-c:v", "libx264", "-c:a", "copy", final_output]
        subprocess.run(ffmpeg_cmd)
        if os.path.exists(final_output):
            return final_output

    print("✅ Lip-Sync Completed Successfully!")
    return raw_output

with gr.Blocks(title="AS-STUDIO-PRO V2.1") as demo:
    gr.Markdown("# AS-STUDIO-PRO V2.1")
    
    with gr.Row():
        with gr.Column():
            video_input = gr.Video(label="Input Video")
            audio_input = gr.Audio(label="Input Audio", type="filepath")
            ratio_input = gr.Dropdown(
                choices=["Original", "16:9", "9:16", "1:1"], 
                value="Original", 
                label="Aspect Ratio"
            )
            btn = gr.Button("Generate Lip-Sync", variant="primary")
        
        with gr.Column():
            video_output = gr.Video(label="Output Video")
            
    btn.click(
        fn=process_lip_sync,
        inputs=[video_input, audio_input, ratio_input],
        outputs=[video_output]
    )

print("🚀 Launching AS-STUDIO-PRO V2.1...")
demo.launch(share=True, debug=True)
